# **Program 9**
Take the Institution name as input. Use Pydantic to define the schema for the desired output and create a custom output parser. Invoke the Chain and Fetch Results. Extract the below Institution related details from Wikipedia: The founder of the Institution. When it was founded. The current branches in the institution . How many employees are working in it. A brief 4-line summary of the institution.

In [2]:
#!pip install langchain cohere langchain-community langchain-cohere wikipedia pydantic -q

## Made by Jhashank Nayan

In [ ]:
import os, json, re
from typing import List
from pydantic import BaseModel, Field
from langchain_cohere import ChatCohere
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser

COHERE_API_KEY = "rninF0VG0cdOJ6ItTgvy8i5AO7daTIiGcJzhbHtG"

# ===== Schema =====
class Info(BaseModel):
    institution_name: str = Field(description="Institution name")
    founder: str = Field(description="Founder")
    founded_year: str = Field(description="Founded year")
    branches: List[str] = Field(description="Branches")
    employees: str = Field(description="Number of employees")
    summary: str = Field(description="4-line summary")

# ===== Parser =====
class Parser(BaseOutputParser):
    def parse(self, text):
        text = re.sub(r"```(?:json)?", "", text).strip()
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if not json_match:
            raise ValueError("No valid JSON found in response")
        data = json.loads(json_match.group())
        # Handle branches as string or list
        branches = data.get("branches", [])
        if isinstance(branches, str):
            branches = [b.strip() for b in re.split(r",|;|\n", branches) if b.strip()]
        data["branches"] = branches
        return Info(**data)

# ===== LLM - Use command-nightly =====
print("🔄 Initializing Cohere LLM (command-nightly)...")
llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-nightly",
    temperature=0.1,
    max_tokens=800
)
print("✅ LLM initialized\n")

# ===== Prompt =====
prompt = PromptTemplate(
    input_variables=["name", "wiki_data"],
    template="""Use this Wikipedia data to extract information:
{wiki_data}

For the institution: {name}

Return ONLY valid JSON with NO markdown, NO code fences:
{{
"institution_name":"",
"founder":"",
"founded_year":"",
"branches":[],
"employees":"",
"summary":"4 lines"
}}

If any value is unknown, write "Not available"."""
)

chain = prompt | llm | Parser()
print("✅ Chain ready\n")

# ===== Run =====
name = "Google"

print(f"🔍 Fetching Wikipedia data for: {name}")

wiki = WikipediaAPIWrapper()
try:
    wiki_data = wiki.run(name)
    if not wiki_data or not wiki_data.strip():
        wiki_data = f"No Wikipedia data available for {name}. Please provide general information."
    print(f"✅ Wikipedia data fetched ({len(wiki_data)} chars)\n")
except Exception as e:
    print(f"⚠️ Wiki fetch failed: {e}\n")
    wiki_data = f"No data found for {name}. Use general knowledge."

print("📝 Processing with Cohere LLM...")

try:
    result = chain.invoke({"name": name, "wiki_data": wiki_data})
    print("\n✅ SUCCESS!\n")
    print("="*60)
    print(f"Institution  : {result.institution_name}")
    print(f"Founder      : {result.founder}")
    print(f"Founded      : {result.founded_year}")
    print(f"Employees    : {result.employees}")
    print(f"\nBranches ({len(result.branches)}):")
    for i, branch in enumerate(result.branches, 1):
        print(f"  {i}. {branch}")
    print(f"\nSummary:\n{result.summary}")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}")
    print(f"Message: {str(e)[:500]}")

🔄 Initializing Cohere LLM (command-nightly)...
✅ LLM initialized

✅ Chain ready

🔍 Fetching Wikipedia data for: Google
⚠️ Wiki fetch failed: HTTPConnectionPool(host='en.wikipedia.org', port=80): Max retries exceeded with url: /w/api.php?list=search&srprop=&srlimit=3&limit=3&srsearch=Google&format=json&action=query (Caused by NewConnectionError("HTTPConnection(host='en.wikipedia.org', port=80): Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network"))

📝 Processing with Cohere LLM...
❌ Error: ConnectError
Message: [Errno 11001] getaddrinfo failed


In [4]:
import os, json, re
from typing import List
from pydantic import BaseModel
from langchain_cohere import ChatCohere
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser

os.environ["COHERE_API_KEY"] = "rninF0VG0cdOJ6ItTgvy8i5AO7daTIiGcJzhbHtG"

# Schema
class Info(BaseModel):
    institution_name: str
    founder: str
    founded_year: str
    branches: List[str]
    employees: str
    summary: str

# Parser
class Parser(BaseOutputParser):
    def parse(self, text):
        data = json.loads(re.search(r"\{.*\}", text, re.S).group())
        if isinstance(data.get("branches"), str):
            data["branches"] = re.split(r",|;|\n", data["branches"])
        return Info(**data)

# LLM + Prompt
llm = ChatCohere(model="command-nightly", temperature=0)

prompt = PromptTemplate.from_template("""
Use this data:
{wiki}

Institution: {name}

Return JSON:
{{
"institution_name":"",
"founder":"",
"founded_year":"",
"branches":[],
"employees":"",
"summary":"4 lines"
}}
""")

chain = prompt | llm | Parser()

# Run
name = "Google"
wiki = WikipediaAPIWrapper().run(name) or ""

result = chain.invoke({"name": name, "wiki": wiki})

print("\n" + "="*50)
print("📊 INSTITUTION DETAILS")
print("="*50)

print(f"🏢 Name       : {result.institution_name}")
print(f"👤 Founder    : {result.founder}")
print(f"📅 Founded    : {result.founded_year}")
print(f"👥 Employees  : {result.employees}")

print("\n🌍 Branches:")
for i, b in enumerate(result.branches, 1):
    print(f"   {i}. {b.strip()}")

print("\n📝 Summary:")
print(result.summary)

print("="*50)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)